# Dataset pilot inspection (issue #5)

Inspect the pilot dataset built from the validated #4 decompilation flow:
Parquet index summary, random side-by-side review of records against their source MP4,
and failure categorization.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pyarrow.parquet as pq

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "scripts"))

dataset_dir = repo_root / "dataset"
records_dir = dataset_dir / "records"
index_path = dataset_dir / "index.parquet"
table = pq.read_table(index_path).to_pylist()
print(f"Index rows: {len(table)}")

status_counts = {}
for row in table:
    status_counts[row["status"]] = status_counts.get(row["status"], 0) + 1
print(f"Status: {status_counts}")

ok_rows = [r for r in table if r["status"] == "ok"]
costs = [r["cost_usd"] for r in ok_rows if r["cost_usd"]]
latencies = [r["latency_seconds"] for r in ok_rows if r["latency_seconds"]]
if costs:
    print(f"Cost: total=${sum(costs):.2f} mean=${sum(costs)/len(costs):.4f}")
if latencies:
    print(f"Latency: mean={sum(latencies)/len(latencies):.0f}s min={min(latencies):.0f}s max={max(latencies):.0f}s")
shots = [r["shot_count"] for r in ok_rows if r["shot_count"]]
print(f"Shots per video: mean={sum(shots)/len(shots):.1f} range={min(shots)}-{max(shots)}")

In [ ]:
import random

random.seed(7)
sample = random.sample(ok_rows, k=min(3, len(ok_rows)))

for row in sample:
    record_dir = records_dir / row["video_id"]
    video_path = record_dir / "video.mp4"
    parsed_path = record_dir / "creative_ir.parsed.json"
    if not parsed_path.exists():
        continue
    ir = json.loads(parsed_path.read_text(encoding="utf-8"))
    print("=" * 80)
    print(f"{row['video_id']}  views={row['views']}  duration={row['duration_seconds']}s  shots={row['shot_count']}")
    print(f"caption: {row['caption'][:120]}")
    hook = ir["observed"].get("hook", {})
    print(f"hook: {hook.get('visual_summary', '')[:140]}")
    concept = ir["inferred"].get("overall_concept", {})
    print(f"concept: {concept.get('premise', '')[:140]}")
    for shot in ir["observed"]["shots"]:
        tr = shot["time_range"]
        texts = [t["text"] for t in shot.get("observed", {}).get("text", {}).get("segments", [])]
        print(f"  {shot['shot_id']} [{tr['start_seconds']:.1f}-{tr['end_seconds']:.1f}s] role={shot.get('inferred', {}).get('semantic_role', '?')} ocr={texts}")
    brief = ir["generation"].get("global_reconstruction_brief", "")
    print(f"brief excerpt: {str(brief)[:200]}")
    data_url = "data:video/mp4;base64," + __import__("base64").b64encode(video_path.read_bytes()).decode()
    display(HTML(f'<video width="270" controls src="{data_url}"></video>'))

In [ ]:
failures = [r for r in table if r["status"] not in ("ok",)]
print(f"Non-ok records: {len(failures)}")
for row in failures:
    print(f"  {row['video_id']}: {row['status']} — {str(row.get('failure_reason'))[:100]}")

print("\n--- model pass / repair statistics across records ---")
repair_total = 0
for record_json in sorted(records_dir.glob("*/record.json")):
    record = json.loads(record_json.read_text(encoding="utf-8"))
    repair_total += record.get("repair_passes", 0) or 0
print(f"JSON repair passes triggered: {repair_total}")